# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/M-Abdullah-Nazeer/Flyrank-Ml-Pipeline/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!pip -q install duckdb
import duckdb, os
import pandas as pd, numpy as np
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"""CREATE OR REPLACE TABLE d AS
SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
page = con.sql("""
WITH past AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS impressions,
         SUM(gsc_clicks) AS clicks,
         SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS position,
         COUNT(*) AS days_with_data,
         MAX(report_date) AS last_day_used
  FROM d
  WHERE gsc_data_available IS TRUE AND report_date <= DATE '2026-03-15' AND gsc_impressions > 0
  GROUP BY 1, 2),
fut AS (
  SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS f_impr, SUM(gsc_clicks) AS f_clicks
  FROM d
  WHERE gsc_data_available IS TRUE AND report_date > DATE '2026-03-15'
  GROUP BY 1, 2)
SELECT p.*, f.f_impr, f.f_clicks
FROM past p LEFT JOIN fut f USING (client_hash_id, content_hash_id)
""").df()

page["ctr"] = page["clicks"] / page["impressions"]
page["pos_bucket"] = pd.cut(page["position"], [0, 3, 6, 10, 20, 1000],
                            labels=["1-3", "4-6", "7-10", "11-20", "21+"])
print(page.shape)

(151981, 11)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule:** among pages with at least 100 impressions and an average position of 20 or better (days 1-15 only), find those whose CTR is below the typical CTR for their position bucket. Score = missed clicks = impressions × (expected CTR − actual CTR). Highest score first.
**Reason code:** `CTR_BELOW_EXPECTED_FOR_POSITION`
**Action label:** `Rewrite title and meta description`

Because the CTR curve is not perfectly smooth across buckets (see Signal 1), pages in the 7-10 bucket are judged against a slightly lower expectation than pages at 11-20, so the rule may under-flag position 7-10.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# SIGNAL 1 (linked to the CTR-fix flag): does CTR fall as position gets worse?
s1 = page.groupby("pos_bucket", observed=True).agg(
    n=("ctr", "size"), impressions=("impressions", "sum"), clicks=("clicks", "sum"))
s1["pooled_ctr"] = (s1["clicks"] / s1["impressions"]).round(4)
display(s1)
print("CTR strictly falls with worse position:", s1["pooled_ctr"].is_monotonic_decreasing)

# expected CTR per position bucket (from days 1-15 only)
expected = s1["pooled_ctr"].to_dict()
page["expected_ctr"] = page["pos_bucket"].map(expected).astype(float)

# SIGNAL 2: does "CTR below expected" persist? (days 16-31 used ONLY to check this, never in the score)
elig = page[(page["impressions"] >= 100) & (page["position"] <= 20) & page["f_impr"].notna()].copy()
elig["gap_ratio"] = elig["ctr"] / elig["expected_ctr"]
elig["gap_group"] = pd.cut(elig["gap_ratio"], [-0.01, 0.5, 0.8, 1.2, 1e9],
                           labels=["much below", "below", "near", "above"])
elig["f_expected_clicks"] = elig["f_impr"] * elig["expected_ctr"]
s2 = elig.groupby("gap_group", observed=True).agg(
    n=("ctr", "size"), f_clicks=("f_clicks", "sum"), f_expected=("f_expected_clicks", "sum"))
s2["future_actual_vs_expected"] = (s2["f_clicks"] / s2["f_expected"]).round(3)
display(s2)

,n,impressions,clicks,pooled_ctr
pos_bucket,,,,
1-3,16840,20573180.0,89967.0,0.0044
4-6,36128,43411368.0,150934.0,0.0035
7-10,35593,19358369.0,57130.0,0.0030
11-20,24888,13164047.0,43983.0,0.0033
21+,37226,30999199.0,42753.0,0.0014


CTR strictly falls with worse position: False


,n,f_clicks,f_expected,future_actual_vs_expected
gap_group,,,,
much below,30736,44424.0,139604.3134,0.318
below,6810,38895.0,69821.8136,0.557
near,6805,50454.0,65678.6608,0.768
above,17856,232537.0,145769.7770,1.595


**Signal 1 (CTR vs position, behind the CTR-fix flag): VERDICT MIXED.** n per bucket: 16,840 (pos 1-3), 36,128 (4-6), 35,593 (7-10), 24,888 (11-20), 37,226 (21+). Pooled CTR runs 0.44%, 0.35%, 0.30%, 0.33%, 0.14%. The big picture holds (top-3 pages get the highest CTR and 21+ the lowest), but it is not strictly falling: 11-20 (0.33%) is slightly above 7-10 (0.30%). So I use the position-bucket CTR as an approximate expectation, not an exact curve.

**Signal 2 (low CTR persists into days 16-31): VERDICT CONFIRMED.** Pages with a low CTR for their position in days 1-15 kept getting fewer clicks than expected in days 16-31. Future actual vs expected clicks: much below 0.318 (n = 30,736), below 0.557 (n = 6,810), near 0.768 (n = 6,805), above 1.595 (n = 17,856). The order is "much below" < "below" < "near" < "above", so the gap is a persistent page trait and not just one-off noise. This supports using the gap as the score.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q = page[(page["impressions"] >= 100) & (page["position"] <= 20)].copy()
q["score"] = (q["impressions"] * (q["expected_ctr"] - q["ctr"])).round(2)
q = q[q["score"] > 0].sort_values("score", ascending=False).reset_index(drop=True)
q["rank"] = q.index + 1
q["reason_code"] = "CTR_BELOW_EXPECTED_FOR_POSITION"
q["action"] = "Rewrite title and meta description"

cols = ["rank", "client_hash_id", "content_hash_id", "score", "reason_code", "action",
        "impressions", "clicks", "ctr", "expected_ctr", "position", "days_with_data"]
os.makedirs("work/outputs", exist_ok=True)
q[cols].head(1000).to_csv("work/outputs/baseline_action_score.csv", index=False)
print(len(q), "pages scored; top 1000 written")
display(q[cols].head(10))

41304 pages scored; top 1000 written


,rank,client_hash_id,content_hash_id,score,reason_code,action,impressions,clicks,ctr,expected_ctr,position,days_with_data
0,1,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,368.60,CTR_BELOW_EXPECTED_FOR_POSITION,Rewrite title and meta description,83772.0,0.0,0.000000,0.0044,0.105274,15
1,2,client_62f4a7e64f5e0096,content_34a70fea29d15f24,306.01,CTR_BELOW_EXPECTED_FOR_POSITION,Rewrite title and meta description,73639.0,18.0,0.000244,0.0044,2.948003,15
2,3,client_62f4a7e64f5e0096,content_7c6373141eae744a,253.01,CTR_BELOW_EXPECTED_FOR_POSITION,Rewrite title and meta description,86860.0,51.0,0.000587,0.0035,5.953765,15
3,4,client_73cda7b4e4f265ea,content_8e1334d6356668e3,203.94,CTR_BELOW_EXPECTED_FOR_POSITION,Rewrite title and meta description,58553.0,1.0,0.000017,0.0035,4.753471,15
4,5,client_20259bd6705d81d4,content_82e35c4845e6c391,202.56,CTR_BELOW_EXPECTED_FOR_POSITION,Rewrite title and meta description,70169.0,29.0,0.000413,0.0033,17.936539,15
5,6,client_62f4a7e64f5e0096,content_1642f339bd6e7c8d,165.32,CTR_BELOW_EXPECTED_FOR_POSITION,Rewrite title and meta description,52378.0,18.0,0.000344,0.0035,4.651991,15
6,7,client_62f4a7e64f5e0096,content_acbcc847f8996314,160.00,CTR_BELOW_EXPECTED_FOR_POSITION,Rewrite title and meta description,83715.0,133.0,0.001589,0.0035,3.518354,15
7,8,client_23a62021009f63c4,content_65c75874a23fca87,152.04,CTR_BELOW_EXPECTED_FOR_POSITION,Rewrite title and meta description,55680.0,15.0,0.000269,0.0030,6.682004,15
8,9,client_62f4a7e64f5e0096,content_36fc1ee501ec072d,150.70,CTR_BELOW_EXPECTED_FOR_POSITION,Rewrite title and meta description,46199.0,11.0,0.000238,0.0035,5.300223,15
9,10,client_62f4a7e64f5e0096,content_945d6ff91386c817,145.94,CTR_BELOW_EXPECTED_FOR_POSITION,Rewrite title and meta description,49314.0,2.0,0.000041,0.0030,8.416089,15


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def wrong_if(r):
    if r["days_with_data"] < 8:
        return f"only {int(r['days_with_data'])} days of data, so the gap may be noise"
    if r["position"] <= 3:
        return "a featured snippet, ad or other SERP feature may be taking the clicks, so the title isn't the cause"
    if r["clicks"] == 0:
        return "zero clicks could mean the impressions come from queries the page doesn't really answer"
    return "position moved during the half, so the average position hides a better rank on some days"

for _, r in q.head(10).iterrows():
    print(f"#{int(r['rank'])}: {r['action']} | why: position {r['position']:.1f}, CTR {r['ctr']:.2%} "
          f"vs expected {r['expected_ctr']:.2%} on {int(r['impressions'])} impressions "
          f"(about {r['score']:.0f} missed clicks) | wrong if: {wrong_if(r)}")

#1: Rewrite title and meta description | why: position 0.1, CTR 0.00% vs expected 0.44% on 83772 impressions (about 369 missed clicks) | wrong if: a featured snippet, ad or other SERP feature may be taking the clicks, so the title isn't the cause
#2: Rewrite title and meta description | why: position 2.9, CTR 0.02% vs expected 0.44% on 73639 impressions (about 306 missed clicks) | wrong if: a featured snippet, ad or other SERP feature may be taking the clicks, so the title isn't the cause
#3: Rewrite title and meta description | why: position 6.0, CTR 0.06% vs expected 0.35% on 86860 impressions (about 253 missed clicks) | wrong if: position moved during the half, so the average position hides a better rank on some days
#4: Rewrite title and meta description | why: position 4.8, CTR 0.00% vs expected 0.35% on 58553 impressions (about 204 missed clicks) | wrong if: position moved during the half, so the average position hides a better rank on some days
#5: Rewrite title and meta descrip

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:** rows with `gap_closed_later = True` were probably noise or fixed themselves, so I'd drop them first. (Name the rank numbers.)
**Leakage check:** all inputs use report_date ≤ 2026-03-15 (the check above says so). No product flags and no label-derived columns went into the score. Days 16-31 were used only to check signals and to review the picks afterwards.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage check: every input came from days 1-15
# Leakage check: every input came from days 1-15
last_used = pd.Timestamp(page["last_day_used"].max())
print("latest day used for inputs:", last_used.date())
assert last_used <= pd.Timestamp("2026-03-15")

# Weak picks: top 10 whose CTR in days 16-31 reached what's expected
t = q.head(10).copy()
t["f_ctr"] = t["f_clicks"] / t["f_impr"]
t["gap_closed_later"] = t["f_ctr"] >= t["expected_ctr"]
display(t[["rank", "ctr", "expected_ctr", "f_ctr", "gap_closed_later"]])

latest day used for inputs: 2026-03-15


,rank,ctr,expected_ctr,f_ctr,gap_closed_later
0,1,0.000000,0.0044,0.016129,True
1,2,0.000244,0.0044,0.000360,False
2,3,0.000587,0.0035,0.000700,False
3,4,0.000017,0.0035,0.000000,False
4,5,0.000413,0.0033,0.000420,False
5,6,0.000344,0.0035,0.001235,False
6,7,0.001589,0.0035,0.001481,False
7,8,0.000269,0.0030,0.000324,False
8,9,0.000238,0.0035,0.000186,False
9,10,0.000041,0.0030,0.000335,False


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.